# World Cup EDA — A Fully Worked Example

**Exploratory Data Analysis (EDA)** is the detective phase of any data project: before you model or build a dashboard, you *look* at the data to understand its shape, spot problems, and find the story. This notebook walks through a complete EDA on the project's database — every cell is filled in and explained, so you can read it like a tutorial.

> The companion notebook `01_eda_worldcup.ipynb` leaves TODOs for **you** to solve. This one (`02`) is the **worked reference** — read it first, then do `01`.

**What you'll learn:**
1. The EDA mindset — always look before you leap
2. Univariate analysis — the distribution of goals (and why it's Poisson)
3. Bivariate analysis — xG vs actual goals (finishing skill = alpha)
4. Categorical comparison — goals across tournaments
5. The small-sample trap — why per-90 rates mislead
6. Correlation — which metrics move together
7. Writing up insights — the actual deliverable

Each section pairs a **plain-English explanation** with **runnable code**. Run cells top-to-bottom (`Shift+Enter`).

## Setup

We pull data straight from PostgreSQL using the project's `v_*` analytical views (the clean layer — see `docs/DATA_CLEANING_SQL.md`). `pd.read_sql(query, engine)` runs SQL and hands back a pandas DataFrame: **SQL does the heavy lifting in the database, pandas does the analysis in Python.** That division of labour is exactly how analysts work in industry.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Walk up to the project root so `from database.db import engine` works
ROOT = Path.cwd()
while not (ROOT / 'database' / 'db.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.figsize'] = (8, 4.5)

from database.db import engine
tours = pd.read_sql('SELECT DISTINCT tournament_label FROM v_player_stats ORDER BY 1', engine)
print('Connected to PostgreSQL. Tournaments loaded:')
print(' · '.join(tours['tournament_label']))

## 1. The EDA mindset — *always look first*

Before any analysis, answer four questions about every dataset:
1. **How big is it?** (`.shape` — rows × columns)
2. **What are the columns and their types?** (`.info()` / `.dtypes`)
3. **What does a row actually look like?** (`.head()`)
4. **What's missing?** (`.isna().sum()`)

Skipping this is the #1 cause of wrong analysis — you compute an average over a column that's half-NULL, or treat a text column as a number. **Look first, always.**

In [ ]:
matches = pd.read_sql("""
    SELECT tournament_label, stage, home_team, away_team,
           home_score, away_score, kickoff_utc
    FROM v_match_results
    WHERE competition = 'WORLD_CUP'
""", engine)

print('Shape (rows, cols):', matches.shape)
print('\nMissing values per column:')
print(matches.isna().sum())
matches.head()

Read the output: a few hundred World Cup matches, no missing scores (because `v_match_results` only includes finished games — the view already did that cleaning for us). One row = one match. Good — we can trust aggregates over this.

## 2. Univariate analysis — the distribution of goals

*Univariate* = one variable at a time. Start with the most basic football question: **how many goals happen in a match?** We'll create a `goals` column and look at its distribution.

The key statistical idea: goals are **rare, independent-ish events over 90 minutes** — the textbook setup for a **Poisson distribution**. A Poisson has one parameter, its mean λ (lambda), and a signature property: **its mean equals its variance**. If our data shows mean ≈ variance, goals really are Poisson — which is *why* the match-prediction model treats scoring as a Poisson process.

In [ ]:
matches['goals'] = matches['home_score'] + matches['away_score']
mean, var = matches['goals'].mean(), matches['goals'].var()

fig, ax = plt.subplots()
ax.hist(matches['goals'], bins=range(0, 11), density=True, alpha=0.65,
        color='#00A86B', edgecolor='white', label='Actual matches')
x = np.arange(0, 11)
ax.plot(x, stats.poisson.pmf(x, mean), 'o-', color='#E0003C', lw=2,
        label=f'Poisson(λ={mean:.2f})')
ax.set(xlabel='Goals per match', ylabel='Probability', title='Goals per match vs. a Poisson model')
ax.legend()
plt.show()

print(f'Mean goals/match     : {mean:.2f}')
print(f'Variance             : {var:.2f}')
print(f'Mean ≈ Variance?     : {"YES — Poisson-like" if abs(mean-var) < 0.6 else "not quite"}')

**Interpretation:** the red Poisson curve tracks the green bars closely, and mean ≈ variance. So a single number λ ≈ 2.6 captures World Cup scoring well. This is genuinely useful: to simulate a match you just draw each team's goals from a Poisson whose λ reflects their strength — which is exactly what `models/tournament_sim.py` does.

> **Quant-finance analog:** modelling goal counts with a Poisson is like modelling rare credit-default events or trade arrivals — same distribution, same one-parameter simplicity. Recognising *which* distribution your data follows is half of statistical modelling.

## 3. Bivariate analysis — xG vs actual goals (the finishing question)

*Bivariate* = two variables, looking for a relationship. **Expected goals (xG)** measures *chance quality* — the goals an average player "should" score from those shots. Actual goals are what happened. Plotting one against the other reveals **finishing skill**:
- On the diagonal (`goals = xG`): finished exactly as expected.
- **Above** the line: clinical — scored more than the chances were worth (**over-performance**).
- **Below**: wasteful, or unlucky.

We filter to players with ≥270 minutes (≈3 full matches) so the rates aren't noise — more on that in Section 5.

In [ ]:
players = pd.read_sql("""
    SELECT player, team, goals, xg, minutes
    FROM v_player_stats
    WHERE tournament_label = 'WC 2022' AND minutes >= 270 AND xg IS NOT NULL
""", engine)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(players['xg'], players['goals'], alpha=0.5, s=40, color='#00A86B')
lim = max(players['xg'].max(), players['goals'].max()) + 1
ax.plot([0, lim], [0, lim], '--', color='gray', label='goals = xG (as expected)')

# Label the standout finishers
for _, r in players.nlargest(5, 'goals').iterrows():
    ax.annotate(r['player'].split()[0], (r['xg'], r['goals']),
                fontsize=9, xytext=(4, 4), textcoords='offset points')

ax.set(xlabel='Expected Goals (xG)', ylabel='Actual Goals',
       title='WC 2022 — finishing: actual goals vs. expected')
ax.legend()
plt.show()

corr = players['xg'].corr(players['goals'])
print(f'Correlation(xG, goals) = {corr:.3f}  (1.0 = perfect; ~0.8 is a strong, real relationship)')
players['over_performance'] = (players['goals'] - players['xg']).round(2)
print('\nMost clinical finishers (goals minus xG):')
print(players.nlargest(5, 'over_performance')[['player', 'goals', 'xg', 'over_performance']].to_string(index=False))

**Interpretation:** the strong correlation (~0.8) confirms xG is a real predictor of goals — chance quality mostly explains scoring. But the *interesting* players are the ones **off** the line. A player well above the diagonal either has elite finishing or rode hot luck; the `goals - xg` gap is your measure of it.

> **Quant-finance analog:** `goals - xG` is **alpha** — return above the benchmark the model expected. A scout (or a portfolio manager) cares less about raw output than about *output relative to what was expected given the inputs*. This single chart is the most interview-valuable idea in the project: it shows you think in terms of **performance vs. a model baseline**, not raw totals.

## 4. Categorical comparison — goals across tournaments

A *categorical* variable groups rows (here, `tournament_label`). Comparing a numeric summary across categories is the bread-and-butter of dashboards. Let's compare **goals per match** across every tournament in the database — is the World Cup higher-scoring than the Euros?

In [ ]:
by_tour = pd.read_sql("""
    SELECT tournament_label,
           COUNT(*)                              AS matches,
           ROUND(AVG(home_score + away_score), 2) AS goals_per_match
    FROM v_match_results
    GROUP BY tournament_label
    ORDER BY goals_per_match DESC
""", engine)

fig, ax = plt.subplots()
bars = ax.barh(by_tour['tournament_label'], by_tour['goals_per_match'], color='#6D28D9')
ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set(xlabel='Goals per match', title='Goals per match by tournament')
ax.invert_yaxis()
plt.show()
by_tour

**Interpretation:** notice the spread is fairly tight — elite international tournaments cluster around 2.5–3 goals/match. When you compare categories, always ask *"is this difference big enough to matter, or just noise from a small number of matches?"* (The `matches` column is your sample-size sanity check — a tournament with 32 matches has a noisier average than one with 64.)

## 5. The small-sample trap — why per-90 rates mislead

This is the most important *statistical* lesson in sports analytics. **Per-90 rates** (per 90 minutes ≈ one full match) let you compare a starter to a substitute fairly. But they have a trap: a player who scores once in 15 minutes has a per-90 of **6.0** — six goals a game! — which is obviously not their true level. **Small samples produce extreme rates.**

Let's see it: plot goals-per-90 against minutes played. The extreme values all live on the left (low minutes).

In [ ]:
rates = pd.read_sql("""
    SELECT player, minutes, goals_p90
    FROM v_player_stats
    WHERE tournament_label = 'WC 2022' AND minutes > 0 AND goals_p90 IS NOT NULL
""", engine)

fig, ax = plt.subplots()
ax.scatter(rates['minutes'], rates['goals_p90'], alpha=0.45, color='#E8C547', edgecolor='black', linewidth=0.3)
ax.axvline(270, color='#E0003C', ls='--', label='270-min threshold (≈3 matches)')
ax.set(xlabel='Minutes played', ylabel='Goals per 90',
       title='The small-sample trap: extreme per-90 rates come from tiny minutes')
ax.legend()
plt.show()

# The highest per-90 scorers — note how many got there in very few minutes.
print('Highest goals/90 (sorted by minutes — watch the low-minute inflation):')
hot = rates[rates['goals_p90'] > 0.9].sort_values('minutes')
print(hot[['player', 'minutes', 'goals_p90']].head(8).to_string(index=False))
print('\nThe lesson: a player at ~1.0 goals/90 in <200 minutes is small-sample noise;\n'
      'the same rate sustained over 600+ minutes (e.g. Mbappé) is a real signal.')

**Interpretation:** the wild per-90 values cluster at low minutes and vanish as minutes grow — the cloud narrows toward the true scoring rate (~0.3–0.5/90 for forwards). This is **regression to the mean**: extreme early results pull back toward the average with more data. The fix is the dashed line — a **minimum-minutes filter** (used throughout the app's leaderboards) so you rank *reliable* rates, not noise.

> **Quant-finance analog:** a fund up 80% over one week isn't an 80%/week fund — short windows exaggerate. Same statistics, same discipline: never trust a rate without checking the sample size behind it.

## 6. Correlation — which metrics move together

A **correlation matrix** shows, at a glance, which numeric variables rise and fall together (+1), move oppositely (−1), or are unrelated (0). It's the fastest way to understand a wide table — and to spot **redundant** features before modelling (two metrics correlated at 0.95 carry nearly the same information).

In [ ]:
metrics = pd.read_sql("""
    SELECT goals, assists, xg, shots, key_passes, pressures, minutes
    FROM v_player_stats
    WHERE tournament_label = 'WC 2022' AND minutes >= 270
""", engine)

corr = metrics.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation of player metrics (WC 2022)')
plt.tight_layout()
plt.show()

**How to read it:** scan for the deep greens (strong positive). You'll see `goals`–`xg`–`shots` form a tight attacking cluster (they measure the same underlying thing — shooting), while `pressures` sits more on its own (a defensive metric). For a model, you wouldn't feed all three of goals/xG/shots — they're redundant; pick the most predictive (xG) and drop the rest. **Correlation analysis is how you choose features without guessing.**

> Caution: **correlation is not causation.** Players with more pressures might score less simply because defenders both press more *and* shoot less — the link is their position, not a cause. Always ask what third variable might explain a correlation.

## 7. Write-up — the actual deliverable

An EDA isn't charts; it's the **3–5 sentences of insight** you'd put at the top of a report. That's what a manager reads. Here's the model write-up for everything above — *this* is the skill that gets you hired:

> ### Key findings — World Cup scoring & finishing
>
> 1. **Scoring is Poisson.** Goals per match average ≈ 2.6 with variance ≈ mean, so a single rate λ models World Cup scoring well — the basis for the tournament simulator.
> 2. **xG strongly predicts goals (r ≈ 0.8),** confirming chance quality drives output. The analytically interesting players are the *exceptions* — those who out-score their xG (clinical finishing = alpha over the model baseline).
> 3. **Per-90 rates need a minutes filter.** Extreme rates are an artifact of tiny samples; a ≥270-minute threshold removes the noise and gives a fair, reliable leaderboard.
> 4. **Attacking metrics are redundant** (goals/xG/shots correlate ~0.8–0.9); a model should use xG and drop the rest to avoid double-counting the same signal.
>
> *Recommendation:* rank players on **xG-adjusted output per 90, minimum 270 minutes** — it rewards genuine, repeatable contribution over small-sample flukes.

---

### Your turn — exercises to cement it

1. Re-run Section 3 for **EURO 2024** instead of WC 2022 (change one string). Are the same players clinical?
2. In Section 4, add a column for **goals from the knockout stage only** (`WHERE stage <> 'GROUP_STAGE'`). Do games get tighter (fewer goals) in the knockouts?
3. Build a new bivariate chart: **assists vs. key passes**. Is creating chances (key passes) a good predictor of assists? What does the correlation tell you?
4. Pick any finding above and write your *own* one-sentence insight for it.

When you're comfortable here, open `01_eda_worldcup.ipynb` and do its guided TODOs — that notebook makes *you* rediscover two real data bugs.